In [0]:
# Import Required Libraries
from pyspark.sql.functions import *

In [0]:
# Load Bronze Tables 
crm = spark.table("crm_customers_100_k")
billing = spark.table("billing_transactions_300_k")
usage = spark.table("customer_usage_500_k")
support = spark.table("support_tickets_200_k")
churn = spark.table("churn_labels_100_k")

In [0]:
spark.sql("SHOW TABLES").show(truncate=False)

+--------+--------------------------+-----------+
|database|tableName                 |isTemporary|
+--------+--------------------------+-----------+
|default |billing_transactions_300_k|false      |
|default |churn_labels_100_k        |false      |
|default |crm_customers_100_k       |false      |
|default |customer_usage_500_k      |false      |
|default |silver_billing            |false      |
|default |silver_churn              |false      |
|default |silver_customers          |false      |
|default |silver_support            |false      |
|default |silver_usage              |false      |
|default |support_tickets_200_k     |false      |
+--------+--------------------------+-----------+



In [0]:
# Verify Row Count 
print("CRM Customers:", crm.count())

print("Billing Transactions:", billing.count())

print("Customer Usage:", usage.count())

print("Support Tickets:", support.count())

print("Churn Labels:", churn.count())

CRM Customers: 100000
Billing Transactions: 300000
Customer Usage: 500000
Support Tickets: 200000
Churn Labels: 100000


In [0]:
# Display all Sample Records (Inspect the raw data) 
display(crm.limit(10))

CustomerID,JoinDate,Segment,Region,Age,SubscriptionType
100000,2020-06-27,Premium,USA,57,Monthly
100001,2024-03-27,Basic,Germany,38,Annual
100002,2023-08-02,Premium,Germany,64,Annual
100003,2022-05-27,Basic,Canada,38,Annual
100004,2022-05-16,Basic,Canada,39,Monthly
100005,2024-09-13,Basic,UK,63,Annual
100006,2020-06-20,Basic,USA,45,Annual
100007,2023-10-26,Basic,Canada,48,Monthly
100008,2021-02-06,Premium,Germany,65,Monthly
100009,2020-07-07,Basic,Canada,57,Monthly


In [0]:
display(billing.limit(10))

TransactionID,CustomerID,TransactionDate,Revenue,Product
T0,139260,2024-02-06,2291.66,Plan B
T1,193679,2025-04-12,4997.86,Plan B
T2,131483,2025-12-12,2892.57,Plan A
T3,118325,2024-10-01,2082.69,Plan A
T4,180032,2025-02-21,2756.25,Plan C
T5,127085,2025-06-22,2263.74,Plan B
T6,196622,2025-07-21,7108.16,Plan B
T7,149639,2025-06-24,3358.92,Plan A
T8,173360,2024-09-18,1776.46,Plan C
T9,196749,2024-12-11,3802.08,Plan B


In [0]:
display(usage.limit(10))

EventID,CustomerID,LoginDate,SessionMinutes
E0,171006,2025-11-25,65
E1,149238,2025-04-22,16
E2,158393,2025-08-01,172
E3,135509,2025-07-22,105
E4,135801,2025-04-22,40
E5,176792,2024-03-21,82
E6,158905,2024-08-26,146
E7,111404,2025-05-04,138
E8,189103,2025-09-29,136
E9,125993,2025-02-21,129


In [0]:
display(support.limit(10))

TicketID,CustomerID,TicketDate,Category,Status
TK0,103995,2025-01-07,Technical,Closed
TK1,134936,2025-08-20,Billing,Closed
TK2,132070,2024-11-28,Login,Open
TK3,161308,2024-02-29,Technical,Closed
TK4,157275,2025-10-17,Billing,Closed
TK5,165885,2025-01-19,Account,Closed
TK6,199202,2024-09-01,Technical,Open
TK7,184558,2024-08-07,Technical,Closed
TK8,145583,2025-07-14,Technical,Closed
TK9,197379,2025-07-21,Login,Closed


In [0]:
display(churn.limit(10))

CustomerID,Churn
100000,0
100001,1
100002,0
100003,0
100004,0
100005,0
100006,1
100007,0
100008,0
100009,0


In [0]:
# View Table Schema (Understand the data types before cleaning)  

crm.printSchema()

billing.printSchema()

usage.printSchema()

support.printSchema()

churn.printSchema()

root
 |-- CustomerID: long (nullable = true)
 |-- JoinDate: date (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- SubscriptionType: string (nullable = true)

root
 |-- TransactionID: string (nullable = true)
 |-- CustomerID: long (nullable = true)
 |-- TransactionDate: date (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- Product: string (nullable = true)

root
 |-- EventID: string (nullable = true)
 |-- CustomerID: long (nullable = true)
 |-- LoginDate: date (nullable = true)
 |-- SessionMinutes: long (nullable = true)

root
 |-- TicketID: string (nullable = true)
 |-- CustomerID: long (nullable = true)
 |-- TicketDate: date (nullable = true)
 |-- Category: string (nullable = true)
 |-- Status: string (nullable = true)

root
 |-- CustomerID: long (nullable = true)
 |-- Churn: long (nullable = true)



In [0]:
# Check Missing Values 
# CRM 

crm.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in crm.columns
]).show()

+----------+--------+-------+------+---+----------------+
|CustomerID|JoinDate|Segment|Region|Age|SubscriptionType|
+----------+--------+-------+------+---+----------------+
|         0|       0|      0|     0|  0|               0|
+----------+--------+-------+------+---+----------------+



In [0]:
# Billing 
billing.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in billing.columns
]).show()

+-------------+----------+---------------+-------+-------+
|TransactionID|CustomerID|TransactionDate|Revenue|Product|
+-------------+----------+---------------+-------+-------+
|            0|         0|              0|      0|      0|
+-------------+----------+---------------+-------+-------+



In [0]:
# Uages 
usage.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in usage.columns
]).show()

+-------+----------+---------+--------------+
|EventID|CustomerID|LoginDate|SessionMinutes|
+-------+----------+---------+--------------+
|      0|         0|        0|             0|
+-------+----------+---------+--------------+



In [0]:
# Support 
support.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in support.columns
]).show()

+--------+----------+----------+--------+------+
|TicketID|CustomerID|TicketDate|Category|Status|
+--------+----------+----------+--------+------+
|       0|         0|         0|       0|     0|
+--------+----------+----------+--------+------+



In [0]:
# Churn 

churn.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in churn.columns
]).show()

+----------+-----+
|CustomerID|Churn|
+----------+-----+
|         0|    0|
+----------+-----+



In [0]:
# Check Duplicate Records 
# CRM 

print("CRM duplicates:")

print(
    crm.count() -
    crm.dropDuplicates().count()
)

CRM duplicates:
0


In [0]:
# Billing 
print(
    billing.count() -
    billing.dropDuplicates().count()
)

0


In [0]:
# Usage 
print(
    usage.count() -
    usage.dropDuplicates().count()
)

0


In [0]:
# Support 
print(
    support.count() -
    support.dropDuplicates().count()
)

0


In [0]:
# Churn 
print(
    churn.count() -
    churn.dropDuplicates().count()
)

0


In [0]:
# Generate Summary Statistics (For numeric columns) 

# CRM 
crm.describe().show()

+-------+-----------------+-------+------+------------------+----------------+
|summary|       CustomerID|Segment|Region|               Age|SubscriptionType|
+-------+-----------------+-------+------+------------------+----------------+
|  count|           100000| 100000|100000|            100000|          100000|
|   mean|         149999.5|   NULL|  NULL|          43.54615|            NULL|
| stddev|28867.65779668774|   NULL|  NULL|15.029634362704593|            NULL|
|    min|           100000|  Basic|Canada|                18|          Annual|
|    max|           199999|Premium|   USA|                69|         Monthly|
+-------+-----------------+-------+------+------------------+----------------+



In [0]:
# Billing 
billing.describe().show()

+-------+-------------+------------------+------------------+-------+
|summary|TransactionID|        CustomerID|           Revenue|Product|
+-------+-------------+------------------+------------------+-------+
|  count|       300000|            300000|            300000| 300000|
|   mean|         NULL|149928.04689333335|3202.5515133999775|   NULL|
| stddev|         NULL| 28837.25407296847|1598.6721452415077|   NULL|
|    min|           T0|            100000|             27.36| Plan A|
|    max|       T99999|            199999|          16235.23| Plan C|
+-------+-------------+------------------+------------------+-------+



In [0]:
# Usage 
usage.describe().show()

+-------+-------+------------------+------------------+
|summary|EventID|        CustomerID|    SessionMinutes|
+-------+-------+------------------+------------------+
|  count| 500000|            500000|            500000|
|   mean|   NULL|      149985.14105|         90.114846|
| stddev|   NULL|28881.074819019326|51.669559006081506|
|    min|     E0|            100000|                 1|
|    max| E99999|            199999|               179|
+-------+-------+------------------+------------------+



In [0]:
# Support 
support.describe().show()

+-------+--------+------------------+---------+------+
|summary|TicketID|        CustomerID| Category|Status|
+-------+--------+------------------+---------+------+
|  count|  200000|            200000|   200000|200000|
|   mean|    NULL|     149976.325835|     NULL|  NULL|
| stddev|    NULL|28855.557229303362|     NULL|  NULL|
|    min|     TK0|            100001|  Account|Closed|
|    max| TK99999|            199999|Technical|  Open|
+-------+--------+------------------+---------+------+



In [0]:
# Churn 
churn.describe().show()

+-------+-----------------+-------------------+
|summary|       CustomerID|              Churn|
+-------+-----------------+-------------------+
|  count|           100000|             100000|
|   mean|         149999.5|             0.1796|
| stddev|28867.65779668774|0.38385584983576126|
|    min|           100000|                  0|
|    max|           199999|                  1|
+-------+-----------------+-------------------+



In [0]:
# Distinct Values

# Useful for understanding categorical columns.

# CRM 
crm.select("Segment").distinct().show()

crm.select("Region").distinct().show()

crm.select("SubscriptionType").distinct().show()

+-------+
|Segment|
+-------+
|Premium|
|  Basic|
|   Gold|
+-------+

+-------+
| Region|
+-------+
|    USA|
|Germany|
| Canada|
|     UK|
|  India|
+-------+

+----------------+
|SubscriptionType|
+----------------+
|         Monthly|
|          Annual|
+----------------+



In [0]:
# Billing 
billing.select("Product").distinct().show()

+-------+
|Product|
+-------+
| Plan B|
| Plan A|
| Plan C|
+-------+



In [0]:
# Support 
support.select("Status").distinct().show()

support.select("Category").distinct().show()

+------+
|Status|
+------+
|Closed|
|  Open|
+------+

+---------+
| Category|
+---------+
|Technical|
|  Billing|
|    Login|
|  Account|
+---------+



In [0]:
# Churn 
churn.groupBy("Churn").count().show()

+-----+-----+
|Churn|count|
+-----+-----+
|    0|82040|
|    1|17960|
+-----+-----+



In [0]:
# Data Quality Report

# Create a simple summary. 
print("="*50)

print("Bronze Layer Validation")

print("="*50)

print("CRM:", crm.count())

print("Billing:", billing.count())

print("Usage:", usage.count())

print("Support:", support.count())

print("Churn:", churn.count())

print("="*50)

Bronze Layer Validation
CRM: 100000
Billing: 300000
Usage: 500000
Support: 200000
Churn: 100000


In [0]:
# Bronze Tables 
crm.write.mode("overwrite").saveAsTable("bronze_customers")

billing.write.mode("overwrite").saveAsTable("bronze_billing")

usage.write.mode("overwrite").saveAsTable("bronze_usage")

support.write.mode("overwrite").saveAsTable("bronze_support")

churn.write.mode("overwrite").saveAsTable("bronze_churn")
